# AffectFusion — Day 1 Complete Pipeline (Zero-API, Colab Free Tier)

**Project:** *AffectFusion: A Cross-Attention Transformer Framework for Explainable Multimodal Emotion Recognition from Text and Emoji Semantics*  
**Target:** **60,000 Raw/Cleaned Rows** (2× your 30,000-row final dataset target)  
**Languages:** 40% English (`ENG`: 24,000) | 30% Bangla (`BN`: 18,000) | 30% Banglish (`BNG`: 18,000)  
**Emoji Ratio:** 70% Emoji-Containing (`42,000`) | 30% Non-Emoji (`18,000`)  
**9 Emotion Classes:** `Joy`, `Neutral`, `Anger`, `Sadness`, `Surprise`, `Sarcasm`, `Fear`, `Disgust`, `Anticipation`  

---

### How to Run This Notebook Safely on Colab Free Tier (Zero Memory Overflow & Zero Lost Work)
1. **No API Keys Required:**
   - **Scraping (Member 1):** Uses `yt-dlp` + `youtube-comment-downloader` (YouTube), Arctic Shift / Reddit public JSON (Reddit), Mastodon public hashtag timelines, and Bangla/English news RSS feeds — **100% API-key free**.
   - **AI Labeling (Member 2):** Runs **`Qwen/Qwen2.5-7B-Instruct` (4-bit NF4 quantized)** directly on Colab's free **T4 GPU** (~5.8 GB VRAM) — **100% API-key free** — plus includes a fallback JSON batch exporter if you want to label externally.
2. **Recommended Runtime Split (Saves Free GPU Hours):**
   - Run **Stage 0, Stage 1 (Scraping), and Stage 2 (Cleaning)** on a **CPU Runtime** first.
   - Then switch to **T4 GPU** (`Runtime → Change runtime type → T4 GPU`), re-run **Stage 0** (mounts Drive), and run **Stage 3 (AI Labeling Setup & 100-Row Test)**.
3. **Crash-Proof Checkpointing & Low RAM:**
   - Uses a disk-backed **SQLite database** stored in Google Drive for $O(1)$ deduplication and exact 6-bucket quota tracking (keeps RAM under ~500 MB even at 60,000+ rows).
   - Automatically checkpoints to Google Drive after every stage and every 500 scraped rows. If Colab disconnects, simply re-run — it resumes from the exact row count where it stopped!


## Stage 0: Mount Google Drive, Install Dependencies & Configure Quotas
Run this cell first in **every** Colab session (CPU or GPU).


In [1]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Master project directory on Google Drive (all checkpoints & outputs live here)
BASE_DIR = '/content/drive/MyDrive/AffectFusion_Day1'
DIRS = {
    'base': BASE_DIR,
    'raw': os.path.join(BASE_DIR, '01_raw_shards'),
    'clean': os.path.join(BASE_DIR, '02_cleaned'),
    'label_test': os.path.join(BASE_DIR, '03_ai_labeling_test'),
    'label_batches': os.path.join(BASE_DIR, '04_ai_labeling_checkpoints'),
    'logs': os.path.join(BASE_DIR, 'logs'),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("Google Drive mounted and workspace ready at:", BASE_DIR)


Mounted at /content/drive
Google Drive mounted and workspace ready at: /content/drive/MyDrive/AffectFusion_Day1


In [3]:
!pip install -q yt-dlp youtube-comment-downloader emoji feedparser pandas==2.2.3 requests==2.32.4

In [4]:
import re
import gc
import json
import time
import math
import random
import hashlib
import sqlite3
from datetime import datetime, timezone
from collections import Counter

import requests
import pandas as pd
import emoji
from tqdm.auto import tqdm

# ==============================================================================
# MASTER CONFIGURATION (60,000 ROWS = 2x FINAL 30,000 TARGET)
# ==============================================================================
CONFIG = {
    "TOTAL_TARGET_ROWS": 60_000,
    # Exact 6-bucket quota matrix:
    # 40% ENG (24,000) | 30% BN (18,000) | 30% BNG (18,000)
    # 70% Emoji (42,000) | 30% Non-Emoji (18,000)
    "QUOTAS": {
        ("ENG", 1): 16_800,  # ENG + Emoji
        ("ENG", 0):  7_200,  # ENG + Non-Emoji
        ("BN",  1): 12_600,  # BN  + Emoji
        ("BN",  0):  5_400,  # BN  + Non-Emoji
        ("BNG", 1): 12_600,  # BNG + Emoji
        ("BNG", 0):  5_400,  # BNG + Non-Emoji
    },
    "MIN_CHAR_LEN": 12,
    "MAX_CHAR_LEN": 600,
    "MIN_WORDS": 3,
    "CHECKPOINT_EVERY_N_ROWS": 500,
    "DB_PATH": os.path.join(DIRS['base'], "affectfusion_raw_state.sqlite"),
    "PROGRESS_JSON": os.path.join(DIRS['logs'], "stage1_quota_progress.json"),
    "USED_SOURCES_JSON": os.path.join(DIRS['logs'], "scraped_sources_registry.json"),
    "CLASSES": [
        "Joy", "Neutral", "Anger", "Sadness", "Surprise",
        "Sarcasm", "Fear", "Disgust", "Anticipation"
    ],
    "REQUEST_TIMEOUT": 20,
    "RANDOM_SEED": 42,
}

random.seed(CONFIG["RANDOM_SEED"])

assert sum(CONFIG["QUOTAS"].values()) == CONFIG["TOTAL_TARGET_ROWS"], "Quota sum mismatch!"
print("Target Configuration Loaded:")
for (lang, has_emo), target in CONFIG["QUOTAS"].items():
    tag = "Emoji" if has_emo else "Non-Emoji"
    print(f"  - {lang:3s} | {tag:9s}: {target:>6,}")
print(f"  TOTAL TARGET       : {CONFIG['TOTAL_TARGET_ROWS']:>6,}")


Target Configuration Loaded:
  - ENG | Emoji    : 16,800
  - ENG | Non-Emoji:  7,200
  - BN  | Emoji    : 12,600
  - BN  | Non-Emoji:  5,400
  - BNG | Emoji    : 12,600
  - BNG | Non-Emoji:  5,400
  TOTAL TARGET       : 60,000


### Core Utilities: Language Classifier (`ENG` / `BN` / `BNG`), Emoji Extractor & Disk-Backed Quota Manager
- **Why SQLite on Drive?** Checking duplicates across 60,000+ rows in RAM slows down and risks OOM. SQLite stores unique MD5 hashes on disk with instant indexed lookups and survives Colab disconnects.
- **Language Classification:** Separates Bengali script (`BN`), Romanized Bengali (`BNG`), and English (`ENG`) using Unicode ranges + curated high-precision Banglish/English lexicons.


In [5]:
# ==============================================================================
# 1. EMOJI & LANGUAGE UTILITIES
# ==============================================================================
SKIN_TONE_AND_VS_RE = re.compile(r'[\U0001F3FB-\U0001F3FF\uFE0E\uFE0F]')
BENGALI_CHAR_RE = re.compile(r'[\u0980-\u09FF]')
LATIN_CHAR_RE = re.compile(r'[A-Za-z]')
URL_RE = re.compile(r'(https?://\S+|www\.\S+)', re.IGNORECASE)
TOKEN_RE = re.compile(r"[A-Za-z']+")

# Strong Banglish vocabulary
BANGLISH_STRONG = {
    "bhalo", "valo", "kemon", "korbo", "korche", "korse", "koreche", "hoyeche", "hoise",
    "hobe", "tumi", "apni", "amader", "tomar", "tomader", "amar", "kotha", "kothay",
    "keno", "kobe", "ekhon", "akhon", "manush", "onek", "khub", "darun", "shundor",
    "sundor", "baje", "faltu", "pagol", "haschi", "kanna", "bhoy", "obak", "opekkha",
    "osthir", "joss", "fatafati", "dhonnobad", "dukkho", "kosto", "koshto", "khushi",
    "anondo", "moja", "mojar", "ghrina", "biraktikor", "birokto", "lojja", "lojjajonok",
    "bishwas", "bissash", "shotti", "sotti", "mithya", "mittha", "natok", "bujhlam",
    "bujhsi", "bujhte", "bujhina", "janina", "parina", "parbo", "lagche", "lagse",
    "chilo", "silo", "ache", "ase", "kintu", "tahole", "ebong", "jodi", "tader",
    "shobai", "sobai", "vaiya", "bhaiya", "apu", "bollo", "bolse", "dekhlam", "dekhsi",
    "dekhbo", "dekhun", "shunun", "korun", "bolen", "geche", "gese", "ashche", "asche",
    "ashbe", "korbe", "pabo", "pabe", "dibe", "nibe", "korlam", "korsi", "khaisi",
    "khaise", "kivabe", "kibhabe", "koto", "kichu", "kisu", "sobcheye", "beshi",
    "ektu", "kokhono", "ekdom", "asole", "ashole", "somoy", "shomoy", "ajke", "kalke",
    "porshu", "vabchi", "bhabchi", "vabsi", "nahole", "naile", "sheta", "seta",
    "eita", "oita", "egulo", "ogulo", "ekhane", "okhane", "shekhane", "sekhane",
    "tarpor", "erpor", "shathe", "sathe", "jonno", "jonne", "theke", "moddhe", "modde",
    "upore", "pashe", "samne", "pichone", "bhetore", "shuru", "shesh", "sesh",
    "bangladesh", "desher", "desh", "taka", "bhai", "vai", "dada", "mama", "kaku",
    "chacha", "ammu", "abbu", "ma", "baba", "bon", "chele", "meye", "polapan",
    "choto", "boro", "notun", "puraton", "purano", "shob", "sob", "ekta", "duita",
    "tinta", "kono", "keu", "abar", "mane", "mone", "hoy", "hoye", "chole", "gelo",
    "elo", "dilo", "nilo", "chai", "chay", "chaise", "thik", "thak", "thake",
    "koro", "bolo", "dekho", "shono", "jao", "asho", "nao", "dao", "kharap",
    "boka", "gadha", "chagol", "bal", "sala", "shala", "harami", "chor",
    "batpar", "dhoka", "dhokabaj", "neta", "sorkar", "shorkar", "rajniti", "bichar",
    "shasti", "dabi", "andolon", "protibad", "bortoman", "obostha", "poristhiti",
    "asha", "opekkhay", "ashay", "proti", "shuvo", "ovinandhon",
    "abhinandan", "mubarak", "inshallah", "mashallah", "alhamdulillah", "subhanallah",
    "astaghfirullah", "allah", "rabbi", "doa", "dua", "bhaijan", "huzur", "sirji",
    "ki", "na", "ar", "ta", "to", "je", "er", "te", "ke", "o", "ei", "oi",
    "ami", "tui", "tor", "amra", "ora"
}

BANGLISH_WEAK = {
    "ki", "na", "ar", "ta", "to", "je", "er", "te", "ke", "o", "ei", "oi",
    "ma", "bal", "bon", "ora", "tor", "ami", "tui", "hoy", "nai", "nei",
    "kal", "aj", "din", "raat", "kom", "age", "pore", "mane", "mone", "thik",
    "chai", "bhai", "vai", "mama", "dada", "apu", "bro", "allah", "inshallah",
    "mashallah", "alhamdulillah", "subhanallah", "bangladesh", "desh", "taka"
}
BANGLISH_DECISIVE = BANGLISH_STRONG - BANGLISH_WEAK

ENGLISH_STOPWORDS = {
    "the", "is", "are", "was", "were", "have", "has", "had", "this", "that",
    "with", "from", "they", "their", "there", "what", "when", "where", "which",
    "would", "could", "should", "about", "because", "people", "really", "think",
    "just", "like", "your", "you", "and", "for", "not", "but", "all", "can",
    "will", "been", "being", "into", "more", "very", "some", "them", "then",
    "than", "only", "also", "even", "much", "make", "made", "know", "time",
    "going", "good", "great", "never", "always", "still", "actually", "literally",
    "totally", "definitely", "honestly", "obviously", "seriously", "looking", "forward",
    "cant", "can't", "dont", "don't", "didnt", "didn't", "wont", "won't", "im", "i'm"
}

def extract_emojis(text: str) -> list:
    if not text:
        return []
    found = [item["emoji"] for item in emoji.emoji_list(text)]
    return [SKIN_TONE_AND_VS_RE.sub("", e) for e in found if e]

def detect_language_3way(text: str, source_lang_hint: str = "") -> str:
    """Classify text into 'BN' (Bangla script), 'BNG' (Banglish), 'ENG' (English), or 'OTHER'."""
    if not text:
        return "OTHER"
    no_url = URL_RE.sub(" ", text)
    no_emoji = emoji.replace_emoji(no_url, replace=" ")

    bn_chars = len(BENGALI_CHAR_RE.findall(no_emoji))
    latin_chars = len(LATIN_CHAR_RE.findall(no_emoji))
    total_alpha = bn_chars + latin_chars
    if total_alpha < 5:
        return "OTHER"

    # 1. Bangla Script (BN)
    if bn_chars >= 6 and (bn_chars / max(total_alpha, 1)) >= 0.35:
        return "BN"

    # Reject non-Latin / non-Bengali scripts (Arabic, Hindi/Devanagari, Cyrillic, CJK, etc.)
    non_ascii_non_bn = sum(
        1 for ch in no_emoji
        if ord(ch) > 127 and not ('\u0980' <= ch <= '\u09FF') and ch.isalpha()
    )
    if non_ascii_non_bn > 4 and non_ascii_non_bn > latin_chars * 0.25:
        return "OTHER"

    if latin_chars < 6:
        return "OTHER"

    tokens = [t.lower() for t in TOKEN_RE.findall(no_emoji)]
    if len(tokens) < 3:
        return "OTHER"

    decisive_bng = sum(1 for t in tokens if t in BANGLISH_DECISIVE)
    weak_bng = sum(1 for t in tokens if t in BANGLISH_WEAK)
    eng_hits = sum(1 for t in tokens if t in ENGLISH_STOPWORDS)

    # 2. Banglish (BNG)
    if decisive_bng >= 2:
        return "BNG"
    if decisive_bng >= 1 and (weak_bng >= 1 or source_lang_hint in ("BN", "BNG")):
        return "BNG"
    if weak_bng >= 3 and eng_hits <= 1:
        return "BNG"

    # 3. English (ENG)
    if eng_hits >= 2 or (len(tokens) <= 6 and eng_hits >= 1 and decisive_bng == 0):
        return "ENG"
    if source_lang_hint == "ENG" and eng_hits >= 1 and decisive_bng == 0:
        return "ENG"

    return "OTHER"

def quick_spam_check(text: str) -> bool:
    """Fast filter during scraping so quotas are not polluted by obvious junk."""
    if not text:
        return True
    t = text.strip()
    if len(t) < CONFIG["MIN_CHAR_LEN"] or len(t) > CONFIG["MAX_CHAR_LEN"]:
        return True
    no_url = URL_RE.sub("", t)
    no_emo = emoji.replace_emoji(no_url, replace=" ").strip()
    if len(no_emo.split()) < CONFIG["MIN_WORDS"]:
        return True
    low = t.lower()
    spam_markers = [
        "sub4sub", "subscribe to my channel", "check my channel", "check out my profile",
        "whatsapp me", "telegram.me", "t.me/", "free bitcoin", "crypto signal",
        "i am a bot", "this action was performed automatically", "1xbet", "melbet",
        "click the link in bio", "dm for promo", "earn $", "earn money online",
        "who is watching in 202", "anyone watching in 202", "first comment", "pin me please"
    ]
    if any(m in low for m in spam_markers):
        return True
    if re.search(r'(.)\1{9,}', t):
        return True
    return False

def text_md5(text: str) -> str:
    norm = re.sub(r'\s+', ' ', text.strip().lower())
    return hashlib.md5(norm.encode('utf-8', errors='ignore')).hexdigest()


# ==============================================================================
# 2. DISK-BACKED SQLITE QUOTA MANAGER (LOW RAM + RESUMABLE ON DRIVE)
# ==============================================================================
class QuotaStore:
    def __init__(self, db_path: str, quotas: dict, shard_dir: str, progress_json: str):
        self.db_path = db_path
        self.quotas = quotas
        self.shard_dir = shard_dir
        self.progress_json = progress_json
        self.conn = sqlite3.connect(self.db_path, timeout=60)
        self.conn.execute("PRAGMA journal_mode=WAL;")
        self.conn.execute("PRAGMA synchronous=NORMAL;")
        self._init_schema()
        self.counts = self._load_counts()
        self.pending_buffer = []
        self.shard_idx = self._get_next_shard_idx()

    def _init_schema(self):
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS raw_rows (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                row_hash TEXT UNIQUE NOT NULL,
                text TEXT NOT NULL,
                lang TEXT NOT NULL,
                has_emoji INTEGER NOT NULL,
                emoji_count INTEGER NOT NULL,
                platform TEXT NOT NULL,
                source_id TEXT,
                emotion_hint TEXT,
                collected_at TEXT NOT NULL
            )
        """)
        self.conn.execute("CREATE INDEX IF NOT EXISTS idx_lang_emo ON raw_rows(lang, has_emoji);")
        self.conn.commit()

    def _load_counts(self) -> dict:
        counts = {k: 0 for k in self.quotas.keys()}
        cur = self.conn.execute("SELECT lang, has_emoji, COUNT(*) FROM raw_rows GROUP BY lang, has_emoji")
        for lang, has_emo, cnt in cur.fetchall():
            if (lang, int(has_emo)) in counts:
                counts[(lang, int(has_emo))] = cnt
        return counts

    def _get_next_shard_idx(self) -> int:
        existing = [f for f in os.listdir(self.shard_dir) if f.startswith("shard_") and f.endswith(".jsonl")]
        return len(existing) + 1

    def total_collected(self) -> int:
        return sum(self.counts.values())

    def total_target(self) -> int:
        return sum(self.quotas.values())

    def remaining(self, lang: str, has_emoji: int) -> int:
        return max(0, self.quotas[(lang, has_emoji)] - self.counts[(lang, has_emoji)])

    def lang_remaining(self, lang: str) -> int:
        return self.remaining(lang, 1) + self.remaining(lang, 0)

    def is_complete(self) -> bool:
        return all(self.counts[k] >= self.quotas[k] for k in self.quotas)

    def most_needed_buckets(self) -> list:
        deficits = [(k, self.quotas[k] - self.counts[k]) for k in self.quotas if self.counts[k] < self.quotas[k]]
        deficits.sort(key=lambda x: x[1], reverse=True)
        return deficits

    def try_add(self, text: str, platform: str, source_id: str, emotion_hint: str, source_lang_hint: str = "") -> bool:
        if quick_spam_check(text):
            return False

        emos = extract_emojis(text)
        has_emo = 1 if len(emos) > 0 else 0
        lang = detect_language_3way(text, source_lang_hint=source_lang_hint)
        if lang == "OTHER":
            return False

        if self.counts[(lang, has_emo)] >= self.quotas[(lang, has_emo)]:
            return False

        h = text_md5(text)
        now = datetime.now(timezone.utc).isoformat()
        try:
            self.conn.execute(
                "INSERT INTO raw_rows (row_hash, text, lang, has_emoji, emoji_count, platform, source_id, emotion_hint, collected_at) "
                "VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)",
                (h, text.strip(), lang, has_emo, len(emos), platform, str(source_id)[:120], emotion_hint, now)
            )
            self.counts[(lang, has_emo)] += 1
            self.pending_buffer.append({
                "row_hash": h,
                "text": text.strip(),
                "lang": lang,
                "has_emoji": has_emo,
                "emoji_count": len(emos),
                "platform": platform,
                "source_id": str(source_id)[:120],
                "emotion_hint": emotion_hint,
                "collected_at": now,
            })
            if len(self.pending_buffer) >= CONFIG["CHECKPOINT_EVERY_N_ROWS"]:
                self.checkpoint()
            return True
        except sqlite3.IntegrityError:
            return False

    def checkpoint(self):
        self.conn.commit()
        if self.pending_buffer:
            shard_path = os.path.join(self.shard_dir, f"shard_{self.shard_idx:05d}.jsonl")
            with open(shard_path, "w", encoding="utf-8") as f:
                for rec in self.pending_buffer:
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            self.pending_buffer.clear()
            self.shard_idx += 1

        summary = {
            "updated_at": datetime.now(timezone.utc).isoformat(),
            "total_collected": self.total_collected(),
            "total_target": self.total_target(),
            "percent_complete": round(100.0 * self.total_collected() / max(self.total_target(), 1), 2),
            "buckets": {
                f"{lang}_{'emoji' if e else 'non_emoji'}": {
                    "collected": self.counts[(lang, e)],
                    "target": self.quotas[(lang, e)],
                    "remaining": self.remaining(lang, e)
                }
                for (lang, e) in self.quotas
            }
        }
        with open(self.progress_json, "w", encoding="utf-8") as f:
            json.dump(summary, f, indent=2)
        gc.collect()

    def print_status(self):
        pct = 100.0 * self.total_collected() / max(self.total_target(), 1)
        print(f"\n=== QUOTA STATUS: {self.total_collected():,} / {self.total_target():,} ({pct:.2f}%) ===")
        for (lang, e), target in self.quotas.items():
            cur = self.counts[(lang, e)]
            tag = "Emoji    " if e else "Non-Emoji"
            bar_len = int(20 * cur / max(target, 1))
            bar = "#" * bar_len + "-" * (20 - bar_len)
            print(f"  {lang:3s} | {tag} | [{bar}] {cur:>6,} / {target:>6,} (rem: {target - cur:>5,})")

    def close(self):
        self.checkpoint()
        self.conn.close()

# Initialize QuotaStore (automatically resumes if Drive already has scraped data!)
store = QuotaStore(
    db_path=CONFIG["DB_PATH"],
    quotas=CONFIG["QUOTAS"],
    shard_dir=DIRS["raw"],
    progress_json=CONFIG["PROGRESS_JSON"]
)
store.print_status()



=== QUOTA STATUS: 0 / 60,000 (0.00%) ===
  ENG | Emoji     | [--------------------]      0 / 16,800 (rem: 16,800)
  ENG | Non-Emoji | [--------------------]      0 /  7,200 (rem: 7,200)
  BN  | Emoji     | [--------------------]      0 / 12,600 (rem: 12,600)
  BN  | Non-Emoji | [--------------------]      0 /  5,400 (rem: 5,400)
  BNG | Emoji     | [--------------------]      0 / 12,600 (rem: 12,600)
  BNG | Non-Emoji | [--------------------]      0 /  5,400 (rem: 5,400)


## Stage 1 (Member 1): Zero-API Quota-Driven Scraping Engine (Target: 60,000 Rows)

### How Zero-API Collection Works Across Sources:
1. **YouTube (`yt-dlp` + `youtube-comment-downloader`):** Primary high-volume engine for **ENG**, **BN**, and **BNG** comments with natural emoji usage. Searches targeted videos across all 9 emotions (heavily weighted toward **Sarcasm, Disgust, Fear, Anticipation**) including major Bangladeshi news, comedy/roast, cricket, drama, tech, and mystery channels.
2. **Reddit (`Arctic Shift` public archive + `old.reddit.com` public `.json` fallback):** Zero API key needed; ideal for English and `r/bangladesh` Banglish/English discussions across Sarcasm, Fear, Disgust, Anticipation, and Neutral subreddits.
3. **Mastodon Public Hashtag Timelines (Zero API Key):** Public Fediverse posts from `mastodon.social` filtered by emotion hashtags — strong source of emoji-rich English posts.
4. **Public News Portals (Bangla & English RSS):** Collects public Bangla/English news texts (note: Facebook aggressively blocks unauthenticated Colab scrapers with login walls, so Bangladeshi YouTube news channels like Somoy TV, Jamuna TV, Prothom Alo, Independent TV, and Channel 24 serve as the open, terms-friendly equivalent for public BD comments).

**Guaranteed Completion Loop:** The orchestrator runs in **automated expansion rounds**, dynamically targeting whichever `(Language, Emoji/Non-Emoji)` buckets still have remaining deficits, and saves to Google Drive every 500 rows until **all 60,000 rows** are collected.


In [6]:
import subprocess
from itertools import islice
from youtube_comment_downloader import YoutubeCommentDownloader, SORT_BY_POPULAR
import feedparser
from bs4 import BeautifulSoup

# Track already-scraped video IDs on Drive so resume never repeats work
def load_used_sources() -> set:
    if os.path.exists(CONFIG["USED_SOURCES_JSON"]):
        with open(CONFIG["USED_SOURCES_JSON"], "r", encoding="utf-8") as f:
            return set(json.load(f))
    return set()

def save_used_sources(used: set):
    with open(CONFIG["USED_SOURCES_JSON"], "w", encoding="utf-8") as f:
        json.dump(list(used), f)

USED_SOURCES = load_used_sources()
print(f"Loaded {len(USED_SOURCES):,} previously scraped source IDs from Drive.")

# ==============================================================================
# TARGETED QUERY BANKS (PRIORITIZING SARCASM, DISGUST, FEAR, ANTICIPATION)
# ==============================================================================
YOUTUBE_QUERIES = {
    "ENG": [
        # Sarcasm (Priority)
        ("Sarcasm", "sarcastic roast standup comedy reaction"),
        ("Sarcasm", "worst movie review funny roast"),
        ("Sarcasm", "customer service horror story funny"),
        ("Sarcasm", "corporate cringe linkedin roast comedy"),
        ("Sarcasm", "terrible tech product review roast"),
        ("Sarcasm", "politician caught lying comedy reaction"),
        # Disgust (Priority)
        ("Disgust", "worst hygiene street food reaction"),
        ("Disgust", "disgusting scammer exposed documentary"),
        ("Disgust", "gross food challenge reaction"),
        ("Disgust", "corrupt company scandal exposed"),
        ("Disgust", "worst restaurant kitchen nightmares"),
        # Fear (Priority)
        ("Fear", "true scary horror stories real life"),
        ("Fear", "scariest paranormal caught on camera"),
        ("Fear", "deep ocean thalassophobia terrifying facts"),
        ("Fear", "ai takeover scary warning documentary"),
        ("Fear", "earthquake disaster survival footage"),
        # Anticipation (Priority)
        ("Anticipation", "GTA 6 official trailer reaction hype"),
        ("Anticipation", "upcoming blockbuster movie official trailer 2026"),
        ("Anticipation", "upcoming iphone galaxy leak review waiting"),
        ("Anticipation", "world cup 2026 predictions hype"),
        ("Anticipation", "new anime season trailer release date"),
        # Anger, Sadness, Joy, Surprise, Neutral
        ("Anger", "unfair referee decision worst robbery sports"),
        ("Anger", "inflation cost of living crisis rant"),
        ("Sadness", "most emotional heartbreaking movie scene tribute"),
        ("Sadness", "goodbye pet tribute emotional story"),
        ("Joy", "wholesome surprise reunion heartwarming moments"),
        ("Joy", "greatest comeback goal celebration crowd reaction"),
        ("Surprise", "unbelievable plot twist magic trick reaction"),
        ("Surprise", "shocking discovery science breakthrough news"),
        ("Neutral", "python programming tutorial explained clearly"),
        ("Neutral", "how does quantum computer work documentary"),
    ],
    "BN_BNG": [
        # Bangladeshi YouTube comments naturally yield BOTH Bangla (BN) & Banglish (BNG) with rich emojis!
        # Sarcasm (Priority)
        ("Sarcasm", "bangla funny roast video bangladesh"),
        ("Sarcasm", "bangladesh viral comedy troll reaction"),
        ("Sarcasm", "bangla cricket funny meme roast"),
        ("Sarcasm", "bangladesh viral funny news somoy tv"),
        ("Sarcasm", "bangla stand up comedy satire"),
        ("Sarcasm", "bangladesh politics funny satire comment"),
        # Disgust (Priority)
        ("Disgust", "bangladesh corruption syndicate news jamuna tv"),
        ("Disgust", "dhaka food adulteration mobile court somoy tv"),
        ("Disgust", "bangladesh scammer fraud exposed news"),
        ("Disgust", "dhaka traffic pollution worst condition news"),
        ("Disgust", "bangladesh unfair price hike syndicate protest"),
        # Fear (Priority)
        ("Fear", "bangla bhooter golpo real horror story"),
        ("Fear", "bangladesh earthquake cyclone warning news"),
        ("Fear", "bangla scary mystery true crime documentary"),
        ("Fear", "dhaka fire accident terrifying news footage"),
        ("Fear", "sunday suspense bangla horror story"),
        # Anticipation (Priority)
        ("Anticipation", "bangladesh cricket upcoming match squad prediction"),
        ("Anticipation", "bangla eid natok official trailer upcoming"),
        ("Anticipation", "shakib khan new movie teaser trailer"),
        ("Anticipation", "bangladesh metro rail new line update"),
        ("Anticipation", "new bangla song teaser coming soon"),
        # Anger, Sadness, Joy, Surprise, Neutral
        ("Anger", "bangladesh cricket defeat angry fans reaction"),
        ("Anger", "dhaka protest justice demand breaking news"),
        ("Sadness", "bangla emotional natok sad ending scene"),
        ("Sadness", "bangladesh tragic accident news somoy tv"),
        ("Joy", "bangladesh cricket historic win celebration"),
        ("Joy", "bangla eid festival joy celebration dhaka"),
        ("Surprise", "bangladesh shocking viral news prothom alo"),
        ("Surprise", "unbelievable bangla magic street reaction"),
        ("Neutral", "bangla tech review smartphone comparison"),
        ("Neutral", "bangla history documentary enayet chowdhury"),
    ],
}

EXPANSION_MODIFIERS_ENG = [
    "2026", "2025", "reaction", "viral", "commentary", "exposed", "trailer", "review",
    "compilation", "drama", "debate", "highlights", "interview", "podcast", "shorts"
]
EXPANSION_MODIFIERS_BD = [
    "somoy tv", "jamuna tv", "prothom alo", "independent tv", "channel 24",
    "bd news", "bangla vlog", "bangla natok", "bangla podcast", "bd cricket",
    "2026", "2025", "viral", "live", "update"
]

REDDIT_TARGETS = [
    ("Sarcasm", "sarcasm", "ENG"),
    ("Sarcasm", "facepalm", "ENG"),
    ("Sarcasm", "mildlyinfuriating", "ENG"),
    ("Sarcasm", "antiwork", "ENG"),
    ("Disgust", "iamatotalpieceofshit", "ENG"),
    ("Disgust", "awfuleverything", "ENG"),
    ("Disgust", "scams", "ENG"),
    ("Fear", "nosleep", "ENG"),
    ("Fear", "creepy", "ENG"),
    ("Fear", "oddlyterrifying", "ENG"),
    ("Anticipation", "GTA6", "ENG"),
    ("Anticipation", "movies", "ENG"),
    ("Anticipation", "gaming", "ENG"),
    ("Joy", "MadeMeSmile", "ENG"),
    ("Joy", "wholesomememes", "ENG"),
    ("Sadness", "sad", "ENG"),
    ("Sadness", "GriefSupport", "ENG"),
    ("Anger", "rant", "ENG"),
    ("Surprise", "Unexpected", "ENG"),
    ("Neutral", "AskReddit", "ENG"),
    ("Neutral", "todayilearned", "ENG"),
    ("Anger", "bangladesh", "BNG"),
    ("Neutral", "Dhaka", "BNG"),
]

MASTODON_TAGS = [
    ("Sarcasm", "sarcasm"), ("Disgust", "disgusting"), ("Fear", "horror"),
    ("Anticipation", "excited"), ("Anticipation", "cantwait"), ("Joy", "happy"),
    ("Sadness", "sad"), ("Anger", "angry"), ("Surprise", "shocked"),
    ("Neutral", "tech"), ("Neutral", "news"), ("Neutral", "bangladesh"),
]

RSS_FEEDS = [
    ("Neutral", "BN", "https://www.prothomalo.com/feed/"),
    ("Neutral", "BN", "https://www.bbc.com/bengali/index.xml"),
    ("Neutral", "ENG", "https://www.thedailystar.net/frontpage/rss.xml"),
]

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AffectFusionAcademicBot/1.0"
}

# ==============================================================================
# SCRAPER FUNCTIONS (ALL ZERO-API)
# ==============================================================================
yt_downloader = YoutubeCommentDownloader()

def yt_search_video_ids(query: str, max_results: int = 12) -> list:
    """Use yt-dlp to discover public YouTube video IDs without any API key."""
    cmd = [
        "yt-dlp", "--flat-playlist", "--no-warnings", "--quiet",
        "--print", "%(id)s", f"ytsearch{max_results}:{query}"
    ]
    try:
        out = subprocess.check_output(cmd, timeout=45, text=True, stderr=subprocess.DEVNULL)
        return [line.strip() for line in out.splitlines() if line.strip() and len(line.strip()) == 11]
    except Exception:
        return []

def scrape_youtube_video(video_id: str, emotion_hint: str, lang_hint: str, max_comments: int = 500) -> int:
    """Download comments from a single YouTube video with zero API key."""
    key = f"yt:{video_id}"
    if key in USED_SOURCES:
        return 0
    added = 0
    try:
        gen = yt_downloader.get_comments_from_url(
            f"https://www.youtube.com/watch?v={video_id}",
            sort_by=SORT_BY_POPULAR
        )
        for comment in islice(gen, max_comments):
            txt = comment.get("text", "")
            if store.try_add(txt, "youtube", video_id, emotion_hint, source_lang_hint=lang_hint):
                added += 1
            if store.is_complete():
                break
        USED_SOURCES.add(key)
    except Exception:
        pass
    return added

def scrape_reddit_subreddit(subreddit: str, emotion_hint: str, lang_hint: str, limit: int = 100, before_ts: int = None) -> tuple:
    """Zero-API Reddit comment fetcher using Arctic Shift public API with old.reddit.com JSON fallback."""
    added = 0
    last_ts = before_ts
    # 1. Arctic Shift public archive (zero API key, Colab-friendly)
    try:
        params = {"subreddit": subreddit, "limit": min(limit, 100), "sort": "desc"}
        if before_ts:
            params["before"] = before_ts
        r = requests.get(
            "https://arctic-shift.photon-reddit.com/api/comments/search",
            params=params, headers=HEADERS, timeout=CONFIG["REQUEST_TIMEOUT"]
        )
        if r.status_code == 200:
            data = r.json().get("data", [])
            for item in data:
                body = item.get("body", "")
                cid = item.get("id", "")
                if body and body not in ("[deleted]", "[removed]"):
                    if store.try_add(body, "reddit", f"r/{subreddit}:{cid}", emotion_hint, source_lang_hint=lang_hint):
                        added += 1
                ts = item.get("created_utc")
                if ts:
                    last_ts = int(ts)
                if store.is_complete():
                    break
            return added, last_ts
    except Exception:
        pass

    # 2. Fallback: old.reddit.com public JSON
    try:
        r = requests.get(
            f"https://old.reddit.com/r/{subreddit}/comments.json?limit=100",
            headers=HEADERS, timeout=CONFIG["REQUEST_TIMEOUT"]
        )
        if r.status_code == 200:
            children = r.json().get("data", {}).get("children", [])
            for ch in children:
                d = ch.get("data", {})
                body = d.get("body", "")
                cid = d.get("id", "")
                if body and body not in ("[deleted]", "[removed]"):
                    if store.try_add(body, "reddit", f"r/{subreddit}:{cid}", emotion_hint, source_lang_hint=lang_hint):
                        added += 1
    except Exception:
        pass
    return added, last_ts

def scrape_mastodon_tag(tag: str, emotion_hint: str, max_pages: int = 4) -> int:
    """Zero-API public Mastodon timeline fetcher (high emoji density)."""
    added = 0
    max_id = None
    for _ in range(max_pages):
        if store.is_complete():
            break
        try:
            url = f"https://mastodon.social/api/v1/timelines/tag/{tag}"
            params = {"limit": 40}
            if max_id:
                params["max_id"] = max_id
            r = requests.get(url, params=params, headers=HEADERS, timeout=CONFIG["REQUEST_TIMEOUT"])
            if r.status_code != 200:
                break
            posts = r.json()
            if not posts:
                break
            for p in posts:
                raw_html = p.get("content", "")
                txt = BeautifulSoup(raw_html, "lxml").get_text(" ", strip=True)
                pid = p.get("id", "")
                if store.try_add(txt, "mastodon", f"#{tag}:{pid}", emotion_hint, source_lang_hint="ENG"):
                    added += 1
                max_id = pid
            time.sleep(0.5)
        except Exception:
            break
    return added

def scrape_news_rss() -> int:
    """Zero-API public news portal headlines + summaries (great for Neutral BN/ENG non-emoji)."""
    added = 0
    for emotion_hint, lang_hint, url in RSS_FEEDS:
        if store.is_complete():
            break
        try:
            feed = feedparser.parse(url)
            for entry in feed.entries:
                title = BeautifulSoup(entry.get("title", ""), "lxml").get_text(" ", strip=True)
                summary = BeautifulSoup(entry.get("summary", ""), "lxml").get_text(" ", strip=True)
                combined = f"{title}. {summary}".strip()
                if store.try_add(combined, "news_rss", url, emotion_hint, source_lang_hint=lang_hint):
                    added += 1
        except Exception:
            continue
    return added

print("All zero-API scraper functions initialized.")


Loaded 0 previously scraped source IDs from Drive.
All zero-API scraper functions initialized.


In [ ]:
# ==============================================================================
# STAGE 1 ORCHESTRATOR: LOOPS UNTIL ALL 60,000 ROWS (ALL 6 BUCKETS) ARE FILLED
# ==============================================================================
# Increase MAX_EXPANSION_ROUNDS if needed; safely pauses & resumes from Drive at any point.
MAX_EXPANSION_ROUNDS = 100
COMMENTS_PER_YT_VIDEO = 500

reddit_cursors = {}

def run_quota_scraping_loop():
    round_num = 0
    try:
        # Step A: Quick public News RSS sweep (helps fill Neutral non-emoji fast)
        if not store.is_complete():
            rss_added = scrape_news_rss()
            print(f"[News RSS] Added {rss_added:,} rows.")

        # Step B: Main quota-driven expansion loop (runs until all 6 buckets hit 100%)
        while not store.is_complete() and round_num < MAX_EXPANSION_ROUNDS:
            round_num += 1
            deficits = store.most_needed_buckets()
            print(f"\n>>> ROUND {round_num} | Remaining total: {store.total_target() - store.total_collected():,}")
            print("    Most needed buckets:", [(f"{k[0]}_{'emo' if k[1] else 'txt'}", d) for k, d in deficits[:4]])

            need_eng = store.lang_remaining("ENG") > 0
            need_bn_bng = (store.lang_remaining("BN") + store.lang_remaining("BNG")) > 0

            # 1. Reddit Sweep (ENG + r/bangladesh BNG)
            if need_eng or store.lang_remaining("BNG") > 0:
                sub_sample = random.sample(REDDIT_TARGETS, min(8, len(REDDIT_TARGETS)))
                r_total = 0
                for emo_hint, sub, l_hint in sub_sample:
                    if store.is_complete():
                        break
                    if l_hint == "ENG" and not need_eng:
                        continue
                    prev_ts = reddit_cursors.get(sub)
                    got, new_ts = scrape_reddit_subreddit(sub, emo_hint, l_hint, limit=100, before_ts=prev_ts)
                    reddit_cursors[sub] = new_ts
                    r_total += got
                    time.sleep(0.4)
                print(f"    [Reddit] +{r_total:,} rows")

            # 2. Mastodon Sweep (ENG Emoji-rich posts)
            if store.remaining("ENG", 1) > 0 and round_num <= 8:
                m_total = 0
                for emo_hint, tag in random.sample(MASTODON_TAGS, min(4, len(MASTODON_TAGS))):
                    if store.remaining("ENG", 1) <= 0:
                        break
                    m_total += scrape_mastodon_tag(tag, emo_hint, max_pages=3)
                print(f"    [Mastodon] +{m_total:,} rows")

            # 3. YouTube Targeted Sweep (ENG)
            if need_eng:
                yt_eng_added = 0
                mod = "" if round_num == 1 else " " + random.choice(EXPANSION_MODIFIERS_ENG)
                for emo_hint, base_q in random.sample(YOUTUBE_QUERIES["ENG"], min(5, len(YOUTUBE_QUERIES["ENG"]))):
                    if store.lang_remaining("ENG") <= 0:
                        break
                    vids = yt_search_video_ids(base_q + mod, max_results=6)
                    for vid in vids:
                        if store.lang_remaining("ENG") <= 0:
                            break
                        yt_eng_added += scrape_youtube_video(vid, emo_hint, "ENG", max_comments=COMMENTS_PER_YT_VIDEO)
                print(f"    [YouTube ENG] +{yt_eng_added:,} rows")

            # 4. YouTube Targeted Sweep (BN + BNG — Heavy Priority)
            if need_bn_bng:
                yt_bd_added = 0
                mod = "" if round_num == 1 else " " + random.choice(EXPANSION_MODIFIERS_BD)
                for emo_hint, base_q in random.sample(YOUTUBE_QUERIES["BN_BNG"], min(7, len(YOUTUBE_QUERIES["BN_BNG"]))):
                    if (store.lang_remaining("BN") + store.lang_remaining("BNG")) <= 0:
                        break
                    vids = yt_search_video_ids(base_q + mod, max_results=8)
                    for vid in vids:
                        if (store.lang_remaining("BN") + store.lang_remaining("BNG")) <= 0:
                            break
                        yt_bd_added += scrape_youtube_video(vid, emo_hint, "BNG", max_comments=COMMENTS_PER_YT_VIDEO)
                print(f"    [YouTube BN/BNG] +{yt_bd_added:,} rows")

            # Save checkpoints to Google Drive after every round!
            store.checkpoint()
            save_used_sources(USED_SOURCES)
            store.print_status()

    except KeyboardInterrupt:
        print("\nPaused by user! Saving checkpoint to Google Drive...")
        store.checkpoint()
        save_used_sources(USED_SOURCES)
        store.print_status()

    if store.is_complete():
        print("\nALL 60,000 ROWS COLLECTED AND SAVED TO GOOGLE DRIVE!")
    else:
        print("\nQuota not yet 100% full — re-run this cell to continue seamlessly from your Drive checkpoint.")

# Start scraping!
run_quota_scraping_loop()


[News RSS] Added 81 rows.

>>> ROUND 1 | Remaining total: 59,919
    Most needed buckets: [('ENG_emo', 16800), ('BN_emo', 12600), ('BNG_emo', 12600), ('ENG_txt', 7190)]
    [Reddit] +599 rows
    [Mastodon] +329 rows
    [YouTube ENG] +7,315 rows
    [YouTube BN/BNG] +4,284 rows

=== QUOTA STATUS: 12,608 / 60,000 (21.01%) ===
  ENG | Emoji     | [##------------------]  1,829 / 16,800 (rem: 14,971)
  ENG | Non-Emoji | [###################-]  6,915 /  7,200 (rem:   285)
  BN  | Emoji     | [#-------------------]    809 / 12,600 (rem: 11,791)
  BN  | Non-Emoji | [########------------]  2,298 /  5,400 (rem: 3,102)
  BNG | Emoji     | [--------------------]    293 / 12,600 (rem: 12,307)
  BNG | Non-Emoji | [#-------------------]    464 /  5,400 (rem: 4,936)

>>> ROUND 2 | Remaining total: 47,392
    Most needed buckets: [('ENG_emo', 14971), ('BNG_emo', 12307), ('BN_emo', 11791), ('BNG_txt', 4936)]
    [Reddit] +326 rows
    [Mastodon] +59 rows
    [YouTube ENG] +1,144 rows
    [YouTube BN/B

In [ ]:
# Export Stage 1 Raw Snapshot from SQLite to Parquet on Drive (streaming in 5,000-row chunks -> low RAM)
import pyarrow as pa
import pyarrow.parquet as pq

STAGE1_PARQUET = os.path.join(DIRS['raw'], "stage1_raw_master.parquet")

def export_sqlite_to_parquet(db_path: str, out_parquet: str, chunk_size: int = 5000):
    conn = sqlite3.connect(db_path)
    total = conn.execute("SELECT COUNT(*) FROM raw_rows").fetchone()[0]
    writer = None
    offset = 0
    while offset < total:
        df_chunk = pd.read_sql_query(
            f"SELECT id, row_hash, text, lang, has_emoji, emoji_count, platform, source_id, emotion_hint, collected_at "
            f"FROM raw_rows ORDER BY id LIMIT {chunk_size} OFFSET {offset}",
            conn
        )
        if df_chunk.empty:
            break
        table = pa.Table.from_pandas(df_chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(out_parquet, table.schema)
        writer.write_table(table)
        offset += chunk_size
        del df_chunk, table
        gc.collect()
    if writer:
        writer.close()
    conn.close()
    print(f"Stage 1 master snapshot saved to Drive: {out_parquet} ({total:,} rows)")

export_sqlite_to_parquet(CONFIG["DB_PATH"], STAGE1_PARQUET)


## Stage 2 (Member 2): Chunked Cleaning, Anonymization & Emoji Stream Preprocessing

Processes the Stage 1 Drive snapshot in **5,000-row chunks** (never overflows Colab RAM) and applies:
1. **PII / Handle / URL Anonymization:** Replaces URLs with `<URL>`, `@usernames` with `<USER>`, emails with `<EMAIL>`, and phone numbers with `<PHONE>`.
2. **Spam / Bot / Noise Removal:** Removes bots, promo links, gibberish, and ultra-short/long posts.
3. **Skin-Tone & Modifier Emoji Normalization:** Strips Fitzpatrick skin tones (`🏻-🏿`) and variation selectors (`\uFE0F`) so emojis map cleanly to base semantics.
4. **Exact + Normalized Deduplication:** Removes near-identical duplicates after cleaning.
5. **Stream Separation Features:** Generates `text_only`, `emojis`, `demoji_text`, `emoji_count`, and `emoji_density` — directly ready for Day 3 Feature Engineering.


In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

STAGE1_PARQUET = os.path.join(DIRS['raw'],   "stage1_raw_master.parquet")
STAGE2_PARQUET = os.path.join(DIRS['clean'], "stage2_cleaned_master.parquet")
STAGE2_CSV     = os.path.join(DIRS['clean'], "stage2_cleaned_master.csv")
STAGE2_REPORT  = os.path.join(DIRS['logs'],  "stage2_cleaning_report.json")

EMAIL_RE   = re.compile(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b')
PHONE_RE   = re.compile(r'(?:\+?\d{1,3}[\s.-]?)?(?:\(?\d{3,4}\)?[\s.-]?)\d{3,4}[\s.-]?\d{3,4}')
HANDLE_RE  = re.compile(r'(?<!\w)@[A-Za-z0-9_\.]{2,30}')
MULTISP_RE = re.compile(r'\s+')

def clean_and_anonymize_row(raw_text: str) -> dict:
    """Clean, anonymize PII, normalize emojis, and split text/emoji streams."""
    if not isinstance(raw_text, str):
        return None
    t = raw_text.strip()
    # 1. Anonymize PII
    t = URL_RE.sub("<URL>", t)
    t = EMAIL_RE.sub("<EMAIL>", t)
    t = HANDLE_RE.sub("<USER>", t)
    t = PHONE_RE.sub("<PHONE>", t)

    # 2. Normalize skin tones & variation selectors in-place
    t = SKIN_TONE_AND_VS_RE.sub("", t)
    t = MULTISP_RE.sub(" ", t).strip()

    # 3. Extract normalized emojis & create text-only stream
    emos = [item["emoji"] for item in emoji.emoji_list(t)]
    text_only = MULTISP_RE.sub(" ", emoji.replace_emoji(t, replace=" ")).strip()

    # 4. Quality & spam re-check after cleaning
    text_no_tags = re.sub(r'<(URL|EMAIL|USER|PHONE)>', '', text_only).strip()
    words = text_no_tags.split()
    if len(words) < CONFIG["MIN_WORDS"] or len(text_no_tags) < CONFIG["MIN_CHAR_LEN"]:
        return None

    # 5. Demoji textual descriptions (for early-fusion baseline & feature engineering)
    demoji_tokens = [
        emoji.demojize(e, delimiters=(" ", " ")).replace("_", " ").strip()
        for e in emos
    ]
    demoji_text = ", ".join(demoji_tokens)

    # 6. Normalized hash for post-cleaning deduplication
    clean_hash = hashlib.md5(t.lower().encode("utf-8", errors="ignore")).hexdigest()
    word_count = max(len(words), 1)

    return {
        "clean_hash": clean_hash,
        "clean_text": t,
        "text_only": text_only,
        "emojis": " ".join(emos),
        "demoji_text": demoji_text,
        "has_emoji": 1 if len(emos) > 0 else 0,
        "emoji_count": len(emos),
        "unique_emoji_count": len(set(emos)),
        "word_count": word_count,
        "emoji_density": round(len(emos) / word_count, 4),
    }

def run_stage2_cleaning(in_parquet: str, out_parquet: str, out_csv: str, chunk_size: int = 5000):
    pf = pq.ParquetFile(in_parquet)
    seen_hashes = set()
    writer = None
    csv_header_written = False

    stats = {
        "input_rows": 0,
        "dropped_quality_or_spam": 0,
        "dropped_post_clean_duplicate": 0,
        "kept_rows": 0,
        "bucket_counts": Counter(),
        "platform_counts": Counter(),
    }

    for batch in tqdm(pf.iter_batches(batch_size=chunk_size), desc="Cleaning chunks"):
        df = batch.to_pandas()
        stats["input_rows"] += len(df)
        cleaned_records = []

        for row in df.itertuples(index=False):
            res = clean_and_anonymize_row(row.text)
            if res is None:
                stats["dropped_quality_or_spam"] += 1
                continue
            if res["clean_hash"] in seen_hashes:
                stats["dropped_post_clean_duplicate"] += 1
                continue
            seen_hashes.add(res["clean_hash"])

            rec = {
                "row_id": f"AF_{stats['kept_rows'] + 1:06d}",
                "clean_text": res["clean_text"],
                "text_only": res["text_only"],
                "emojis": res["emojis"],
                "demoji_text": res["demoji_text"],
                "lang": row.lang,
                "has_emoji": res["has_emoji"],
                "emoji_count": res["emoji_count"],
                "unique_emoji_count": res["unique_emoji_count"],
                "word_count": res["word_count"],
                "emoji_density": res["emoji_density"],
                "platform": row.platform,
                "emotion_hint": row.emotion_hint,
            }
            cleaned_records.append(rec)
            stats["kept_rows"] += 1
            stats["bucket_counts"][f"{row.lang}_{'emoji' if res['has_emoji'] else 'non_emoji'}"] += 1
            stats["platform_counts"][row.platform] += 1

        if cleaned_records:
            out_df = pd.DataFrame(cleaned_records)
            table = pa.Table.from_pandas(out_df, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(out_parquet, table.schema)
            writer.write_table(table)

            out_df.to_csv(out_csv, mode="a" if csv_header_written else "w", header=not csv_header_written, index=False)
            csv_header_written = True

            del out_df, table, cleaned_records
        del df
        gc.collect()

    if writer:
        writer.close()

    report = {
        "completed_at": datetime.now(timezone.utc).isoformat(),
        "input_rows": stats["input_rows"],
        "kept_rows": stats["kept_rows"],
        "dropped_quality_or_spam": stats["dropped_quality_or_spam"],
        "dropped_post_clean_duplicate": stats["dropped_post_clean_duplicate"],
        "buckets": dict(stats["bucket_counts"]),
        "platforms": dict(stats["platform_counts"]),
    }
    with open(STAGE2_REPORT, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2)

    print("\n=== STAGE 2 CLEANING COMPLETE ===")
    print(json.dumps(report, indent=2))

    print("\nQuota Check vs. 30k Final Need (and 60k 2x Target):")
    for (lang, e), target_60k in CONFIG["QUOTAS"].items():
        bkey = f"{lang}_{'emoji' if e else 'non_emoji'}"
        have = stats["bucket_counts"].get(bkey, 0)
        target_30k = target_60k // 2
        status = "OK (>=2x)" if have >= target_60k else ("OK (>=1x)" if have >= target_30k else "DEFICIT")
        print(f"  {bkey:14s}: {have:>6,} | 1x need: {target_30k:>6,} | 2x target: {target_60k:>6,} -> {status}")

run_stage2_cleaning(STAGE1_PARQUET, STAGE2_PARQUET, STAGE2_CSV)


## Stage 3 (Member 2): Zero-API Batch AI Annotation & 100-Sample Stability Test

> **Before running Stage 3:** Switch to a **Free T4 GPU** (`Runtime → Change runtime type → T4 GPU`), re-run **Stage 0** (Mount Drive + Config), and jump directly here! All Stage 1 & Stage 2 outputs are already saved on Google Drive.

### Zero-API Design:
- Loads **`Qwen/Qwen2.5-7B-Instruct`** in **4-bit NF4 quantization** (`bitsandbytes`) directly on Colab's free 15 GB T4 GPU (~5.8 GB VRAM used — zero OOM risk).
- Strong native comprehension of **English (`ENG`), Bangla (`BN`), and Romanized Banglish (`BNG`)** plus Unicode emojis.
- Processes **30–35 rows per batch** in **strict JSON format** with:
  - `id`: row ID
  - `label`: one of the **9 emotion classes** (`Joy`, `Neutral`, `Anger`, `Sadness`, `Surprise`, `Sarcasm`, `Fear`, `Disgust`, `Anticipation`)
  - `confidence`: integer `1`–`5`
- Automatically retries missing/malformed IDs, frees CUDA cache after every batch, and checkpoints every batch to Google Drive.
- Runs the **100-Row Day-1 Verification Test** and saves `stage3_sample_100_labeled.csv` to Drive for manual review!


In [ ]:
# Install GPU quantization packages (run on T4 GPU runtime)
!pip install -q -U transformers accelerate bitsandbytes


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

assert torch.cuda.is_available(), (
    "No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU, "
    "re-run Stage 0 (Drive mount + Config), and then run Stage 3."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU Ready: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# Primary model: Qwen2.5-7B-Instruct (excellent ENG/BN/BNG + strict JSON)
MODEL_ID = "Qwen/Qwen2.5-7B-Instruct" if vram_gb >= 12.0 else "Qwen/Qwen2.5-3B-Instruct"
print(f"Loading 4-bit quantized model: {MODEL_ID} ...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print(f"Model loaded! Allocated VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


In [ ]:
# ==============================================================================
# STRICT 9-CLASS EMOTION RUBRIC & BATCH JSON ANNOTATOR (30-50 ROWS / BATCH)
# ==============================================================================
VALID_CLASSES = set(CONFIG["CLASSES"])
STAGE2_PARQUET = os.path.join(DIRS['clean'], "stage2_cleaned_master.parquet")

SYSTEM_PROMPT = """You are an expert multilingual emotion annotator for English (ENG), Bangla (BN), and Romanized Banglish (BNG) social-media comments containing text and emojis.

Classify EACH input comment into EXACTLY ONE of these 9 emotion classes:
1. Joy: happiness, excitement, gratitude, love, celebration, pride, genuine amusement.
2. Neutral: objective facts, plain questions, news statements, or no clear emotional charge.
3. Anger: frustration, outrage, hostility, irritation, hatred, complaints.
4. Sadness: grief, heartbreak, disappointment, loneliness, sympathy, regret.
5. Surprise: shock, astonishment, disbelief, unexpected discovery (positive or negative).
6. Sarcasm: irony, mockery, passive-aggressive praise, OR emoji-text contradiction (e.g., positive words paired with 🙄🤡🙃💀😒 such as "Great, just what I needed 🙄" or "Darun kaj korse 🤡"). ALWAYS prioritize Sarcasm when text and emoji contradict each other ironically.
7. Fear: anxiety, terror, dread, worry, panic, horror, feeling threatened.
8. Disgust: revulsion, nausea, moral repulsion, contempt, calling something gross/corrupt/shameful ("ghrina", "faltu", "lojja", 🤮🤢).
9. Anticipation: looking forward to something, hype, waiting eagerly, hopeful expectation, predictions ("opekkha", "can't wait", "waiting for", 🔥👀⏳).

Also assign a confidence score from 1 to 5:
- 5 = unmistakable emotion
- 4 = clear emotion
- 3 = probable but mild/mixed
- 2 = ambiguous
- 1 = very unclear

OUTPUT RULES (STRICT):
- Return ONLY a valid JSON array of objects — NO markdown fences, NO explanations, NO extra text.
- Every input id MUST appear exactly once.
- Format: [{"id": "AF_000001", "label": "Sarcasm", "confidence": 5}, ...]"""

def build_batch_user_prompt(batch_rows: list) -> str:
    payload = [
        {"id": r["row_id"], "lang": r["lang"], "text": r["clean_text"][:280]}
        for r in batch_rows
    ]
    return (
        "Annotate the following comments. Return ONLY the JSON array:\n"
        + json.dumps(payload, ensure_ascii=False)
    )

def parse_json_array_robust(raw_output: str) -> list:
    """Extract and validate JSON array even if the model wraps output in prose or fences."""
    text = raw_output.strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s*```$', '', text)

    # Attempt 1: Direct array parse
    start = text.find("[")
    end = text.rfind("]")
    if start != -1 and end != -1 and end > start:
        try:
            parsed = json.loads(text[start:end + 1])
            if isinstance(parsed, list):
                return parsed
        except json.JSONDecodeError:
            pass

    # Attempt 2: Recover individual JSON objects via regex
    recovered = []
    for match in re.finditer(r'\{[^{}]+\}', text):
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict) and "id" in obj and "label" in obj:
                recovered.append(obj)
        except json.JSONDecodeError:
            continue
    return recovered

@torch.inference_mode()
def annotate_batch_local_llm(batch_rows: list, max_new_tokens: int = 1800) -> tuple:
    """Annotate a batch of 30-50 rows using local 4-bit Qwen2.5-7B-Instruct."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_batch_user_prompt(batch_rows)},
    ]
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

    t0 = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=None,
        top_p=None,
        pad_token_id=tokenizer.eos_token_id,
    )
    elapsed = time.time() - t0

    gen_ids = outputs[0][inputs["input_ids"].shape[1]:]
    raw_reply = tokenizer.decode(gen_ids, skip_special_tokens=True)

    # Free GPU VRAM immediately to prevent memory creep
    del inputs, outputs, gen_ids
    torch.cuda.empty_cache()
    gc.collect()

    parsed = parse_json_array_robust(raw_reply)
    valid_map = {}
    for item in parsed:
        rid = str(item.get("id", "")).strip()
        lbl = str(item.get("label", "")).strip().capitalize()
        if lbl not in VALID_CLASSES:
            for c in VALID_CLASSES:
                if c.lower() == lbl.lower():
                    lbl = c
                    break
        try:
            conf = int(item.get("confidence", 3))
            conf = max(1, min(5, conf))
        except (TypeError, ValueError):
            conf = 3

        if rid and lbl in VALID_CLASSES:
            valid_map[rid] = {"label": lbl, "confidence": conf}

    return valid_map, elapsed, raw_reply

def annotate_dataframe_with_checkpoints(
    df_input: pd.DataFrame,
    out_jsonl_path: str,
    batch_size: int = 34,
    retry_missing: bool = True
) -> tuple:
    """Run batch annotation with per-batch Google Drive checkpointing & automatic resume."""
    done_map = {}
    if os.path.exists(out_jsonl_path):
        with open(out_jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    rec = json.loads(line)
                    done_map[rec["row_id"]] = rec
        print(f"Resuming from checkpoint: {len(done_map):,} rows already annotated.")

    remaining_df = df_input[~df_input["row_id"].isin(done_map)].reset_index(drop=True)
    rows = remaining_df.to_dict(orient="records")
    total_batches = math.ceil(len(rows) / batch_size)

    batch_stats = []
    for b_idx in range(total_batches):
        chunk = rows[b_idx * batch_size : (b_idx + 1) * batch_size]
        valid_map, elapsed, _ = annotate_batch_local_llm(chunk)

        missing = [r for r in chunk if r["row_id"] not in valid_map]
        if retry_missing and missing:
            rec_map, rec_elapsed, _ = annotate_batch_local_llm(missing, max_new_tokens=900)
            valid_map.update(rec_map)
            elapsed += rec_elapsed

        with open(out_jsonl_path, "a", encoding="utf-8") as f:
            for r in chunk:
                rid = r["row_id"]
                if rid in valid_map:
                    out_rec = dict(r)
                    out_rec["ai_label"] = valid_map[rid]["label"]
                    out_rec["ai_confidence"] = valid_map[rid]["confidence"]
                    f.write(json.dumps(out_rec, ensure_ascii=False) + "\n")
                    done_map[rid] = out_rec

        hit_rate = 100.0 * len(valid_map) / max(len(chunk), 1)
        batch_stats.append((len(chunk), len(valid_map), elapsed))
        print(
            f"  Batch {b_idx + 1}/{total_batches} | Parsed: {len(valid_map)}/{len(chunk)} "
            f"({hit_rate:.1f}%) | Time: {elapsed:.1f}s | Saved to Drive"
        )

    return pd.DataFrame(list(done_map.values())), batch_stats

print("Batch AI Annotation engine ready (30-50 rows/batch, strict JSON, Drive checkpointing).")


In [ ]:
# ==============================================================================
# DAY 1 DELIVERABLE TEST: 100-SAMPLE STRATIFIED STABILITY & ACCURACY CHECK
# ==============================================================================
SAMPLE_100_JSONL = os.path.join(DIRS['label_test'], "sample_100_checkpoint.jsonl")
SAMPLE_100_CSV   = os.path.join(DIRS['label_test'], "stage3_sample_100_labeled.csv")
SAMPLE_100_STATS = os.path.join(DIRS['logs'],       "stage3_sample_100_metrics.json")

# Load ONLY metadata columns first to build a memory-safe stratified 100-row sample
df_clean_meta = pd.read_parquet(STAGE2_PARQUET, columns=["row_id", "lang", "has_emoji"])

# Match exact 40/30/30 language and 70/30 emoji ratio inside the 100-row test
sample_targets = {
    ("ENG", 1): 28, ("ENG", 0): 12,
    ("BN",  1): 21, ("BN",  0):  9,
    ("BNG", 1): 21, ("BNG", 0):  9,
}

selected_ids = []
for (lang, emo), n_req in sample_targets.items():
    sub = df_clean_meta[(df_clean_meta["lang"] == lang) & (df_clean_meta["has_emoji"] == emo)]
    take = min(n_req, len(sub))
    if take > 0:
        selected_ids.extend(sub.sample(n=take, random_state=CONFIG["RANDOM_SEED"])["row_id"].tolist())

if len(selected_ids) < 100:
    extra_pool = df_clean_meta[~df_clean_meta["row_id"].isin(selected_ids)]
    topup = min(100 - len(selected_ids), len(extra_pool))
    if topup > 0:
        selected_ids.extend(extra_pool.sample(n=topup, random_state=CONFIG["RANDOM_SEED"])["row_id"].tolist())

del df_clean_meta
gc.collect()

# Load full columns ONLY for the 100 selected IDs
df_clean_all = pd.read_parquet(STAGE2_PARQUET)
df_sample_100 = df_clean_all[df_clean_all["row_id"].isin(selected_ids)].reset_index(drop=True)
del df_clean_all
gc.collect()

print(f"Running Day-1 100-Sample AI Annotation Test ({len(df_sample_100)} rows, batch_size=34)...")
df_labeled_100, b_stats = annotate_dataframe_with_checkpoints(
    df_sample_100,
    out_jsonl_path=SAMPLE_100_JSONL,
    batch_size=34,
    retry_missing=True
)

df_labeled_100.to_csv(SAMPLE_100_CSV, index=False)

total_time = sum(x[2] for x in b_stats)
recovery_rate = 100.0 * len(df_labeled_100) / max(len(df_sample_100), 1)
est_hours_30k = (total_time * 300) / 3600.0 if total_time > 0 else 0.0

metrics = {
    "tested_rows": len(df_sample_100),
    "successfully_labeled": len(df_labeled_100),
    "json_recovery_rate_pct": round(recovery_rate, 2),
    "total_runtime_sec": round(total_time, 1),
    "est_hours_for_30k_on_t4": round(est_hours_30k, 2),
    "class_distribution": df_labeled_100["ai_label"].value_counts().to_dict(),
    "confidence_distribution": {str(k): int(v) for k, v in df_labeled_100["ai_confidence"].value_counts().sort_index().items()},
}
with open(SAMPLE_100_STATS, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n=== 100-SAMPLE TEST COMPLETE ===")
print(json.dumps(metrics, indent=2))
print(f"\nSaved labeled 100-row sample to Google Drive for manual review: {SAMPLE_100_CSV}")

display(df_labeled_100[["row_id", "lang", "has_emoji", "clean_text", "ai_label", "ai_confidence"]].head(15))


## Stage 4 (Ready for Day 2): Full-Scale Batch Labeling Launcher & External Batch Exporter

Once you inspect `stage3_sample_100_labeled.csv` and verify label accuracy:
- **Option A (Fully Automated on Colab Free T4 GPU — Zero API):** Uncomment and run on Day 2 in chunks of 5,000 rows per session (saves after every batch so GPU disconnects never lose work).
- **Option B (Optional Speed Multiplier):** Export 50-row prompt `.txt` files to Google Drive if both team members want to run extra batches in parallel via free web chat tools without an API key.


In [ ]:
# OPTION A (Day 2): Run full-scale zero-API labeling on T4 GPU (uncomment on Day 2!)
# FULL_LABEL_CHECKPOINT = os.path.join(DIRS['label_batches'], "day2_full_labeled_checkpoint.jsonl")
# FULL_LABEL_PARQUET    = os.path.join(DIRS['label_batches'], "day2_full_labeled_master.parquet")
#
# df_for_day2 = pd.read_parquet(STAGE2_PARQUET)
# CHUNK_START, CHUNK_END = 0, 5000   # Run 5,000 rows per Colab GPU session (resumes automatically)
# df_labeled_chunk, _ = annotate_dataframe_with_checkpoints(
#     df_for_day2.iloc[CHUNK_START:CHUNK_END],
#     out_jsonl_path=FULL_LABEL_CHECKPOINT,
#     batch_size=35,
#     retry_missing=True
# )
# df_labeled_chunk.to_parquet(FULL_LABEL_PARQUET, index=False)
# print("Saved Day 2 checkpoint to:", FULL_LABEL_PARQUET)

# OPTION B: Export ready-to-paste 50-row JSON prompts to Drive (optional parallel helper)
def export_prompt_batches(in_parquet: str, out_dir: str, num_batches: int = 5, batch_size: int = 50):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_parquet(in_parquet, columns=["row_id", "lang", "clean_text"]).head(num_batches * batch_size)
    for i in range(num_batches):
        sub = df.iloc[i * batch_size : (i + 1) * batch_size].to_dict(orient="records")
        if not sub:
            break
        prompt_str = SYSTEM_PROMPT + "\n\n" + build_batch_user_prompt(sub)
        out_path = os.path.join(out_dir, f"prompt_batch_{i+1:04d}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(prompt_str)
    print(f"Exported {num_batches} ready-to-paste 50-row prompt files to: {out_dir}")

# Uncomment to test exporting 5 sample 50-row prompt files to Drive:
# export_prompt_batches(STAGE2_PARQUET, os.path.join(DIRS['label_batches'], "manual_prompt_batches"), num_batches=5)
print("Stage 4 helpers loaded and ready for Day 2.")
